In [0]:
%sql
    
CREATE OR REPLACE TABLE dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.BOSQL_MI_Mapping_V3 AS    
WITH raw_data AS (
  SELECT DISTINCT
    upper(trim(table)) as BO_TABLE,
    CASE 
      WHEN upper(SOURCE) LIKE 'ALIAS%' AND base_table IS NOT NULL AND base_table != '' 
      THEN upper(trim(base_table)) 
      ELSE upper(trim(TABLE)) 
    END AS SOURCE_TABLE,
    upper(trim(SCHEMA)) as SCHEMA,
    CASE
      WHEN upper(SOURCE) LIKE 'ALIAS%'
      THEN upper(trim(regexp_replace(regexp_replace(SOURCE, '(?i)^Alias\\s*', ''), '[()\\[\\]{}]', '')))
      ELSE upper(trim(SOURCE))
    END AS SOURCE
  FROM dataplatform01_modelling_dev_catalog_01.bo_universes_excel_sheet_imports.v3_extraction
  UNION
  SELECT DISTINCT
    upper(trim(table)) as BO_TABLE,
    CASE 
      WHEN upper(SOURCE) LIKE 'ALIAS%' AND base_table IS NOT NULL AND base_table != '' 
      THEN upper(trim(base_table)) 
      ELSE upper(trim(TABLE)) 
    END AS SOURCE_TABLE,
    upper(trim(SCHEMA)) as SCHEMA,
    CASE
      WHEN upper(SOURCE) LIKE 'ALIAS%'
      THEN upper(trim(regexp_replace(regexp_replace(SOURCE, '(?i)^Alias\\s*', ''), '[()\\[\\]{}]', '')))
      ELSE upper(trim(SOURCE))
    END AS SOURCE
  FROM dataplatform01_modelling_dev_catalog_01.bo_universes_excel_sheet_imports.v3_extraction_remaining
),
-- Level 1: Deduplicate by (BO_TABLE, SOURCE_TABLE, SCHEMA) — keep highest priority SOURCE
ranked_source AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY BO_TABLE, SOURCE_TABLE, SCHEMA
      ORDER BY
        CASE 
          WHEN SOURCE = 'SYMPHONY' THEN 1
          WHEN SOURCE = 'ORACLE FINANCE' THEN 2
          WHEN SOURCE = 'ORACLE FINANCE TABLE' THEN 3
          WHEN SOURCE = 'ORACLE FINANCE VIEW' THEN 4
          WHEN SOURCE = 'ORACLEFINANCE' THEN 5
          WHEN SOURCE = 'ORACLE FINANCE MATERIALIZED VIEW' THEN 6
          WHEN SOURCE = 'ORACLE FINACNE' THEN 7
          WHEN SOURCE = 'DW (FACT)' THEN 8
          WHEN SOURCE = 'DW' THEN 9
          WHEN SOURCE = 'DW (DIMENSION)' THEN 10
          WHEN SOURCE = 'DW (VIEW)' THEN 11
          WHEN SOURCE = 'INFORMATICA' THEN 12
          WHEN SOURCE = 'SYSTEM TABLE' THEN 13
          WHEN SOURCE = 'DERIVED' THEN 14
          WHEN SOURCE IN ('NA', 'NOT AVAILABLE') THEN 15
          ELSE 99
        END
    ) AS rn1
  FROM raw_data
),
deduped_source AS (
  SELECT BO_TABLE, SOURCE_TABLE, SCHEMA, SOURCE
  FROM ranked_source
  WHERE rn1 = 1
),
-- Level 2: Deduplicate by (BO_TABLE, SOURCE_TABLE) — keep highest priority SCHEMA
ranked_schema AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY BO_TABLE, SOURCE_TABLE
      ORDER BY
        CASE 
          WHEN SCHEMA = 'ORADMART1' THEN 1
          WHEN SCHEMA = 'ORABUP0' THEN 2
          WHEN SCHEMA = 'AR' THEN 3
          WHEN SCHEMA = 'GL' THEN 4
          WHEN SCHEMA = 'APPS' THEN 5
          WHEN SCHEMA = 'AP' THEN 6
          WHEN SCHEMA = 'CUST' THEN 7
          WHEN SCHEMA = 'FA' THEN 8
          WHEN SCHEMA = 'APPLSYS' THEN 9
          WHEN SCHEMA = 'PO' THEN 10
          WHEN SCHEMA = 'HR' THEN 11
          WHEN SCHEMA = 'XLA' THEN 12
          WHEN SCHEMA = 'JTF' THEN 13
          WHEN SCHEMA = 'INV' THEN 14
          WHEN SCHEMA = 'ZX' THEN 15
          WHEN SCHEMA = 'OKC' THEN 16
          WHEN SCHEMA = 'ICX' THEN 17
          WHEN SCHEMA = 'SYS' THEN 18
          WHEN SCHEMA = 'ORABOFP' THEN 19
          WHEN SCHEMA = 'GBRELS1' THEN 20
          WHEN SCHEMA IN ('NOT AVAILABLE', 'NOT AVILABLE') THEN 21
          ELSE 99
        END
    ) AS rn2
  FROM deduped_source
)
SELECT BO_TABLE, SOURCE_TABLE, SCHEMA, SOURCE
FROM ranked_schema
WHERE rn2 = 1
ORDER BY BO_TABLE ASC, SOURCE_TABLE ASC



In [0]:
%sql
     
-- Recursive trace: when SOURCE_SCHEMA = 'ORADMART1', follow the chain to find real source
WITH RECURSIVE source_dict AS (
  SELECT DISTINCT
    upper(trim(TARGET_SCHEMA)) as TARGET_SCHEMA,
    upper(trim(TARGET_TABLE)) as TARGET_TABLE,
    upper(trim(SOURCE_SCHEMA)) as SOURCE_SCHEMA,
    upper(trim(SOURCE_TABLE)) as SOURCE_TABLE
  FROM dataplatform01_modelling_dev_catalog_01.bo_universes_excel_sheet_imports.mi_dwh_data_dictionary
  WHERE SOURCE_SCHEMA IS NOT NULL 
    AND upper(trim(SOURCE_SCHEMA)) NOT IN ('CREATED AND DROPPED', 'SYS')
    AND TARGET_TABLE IS NOT NULL
    AND upper(trim(TARGET_SCHEMA)) NOT IN ('CREATED AND DROPPED')
),
lineage AS (
  -- Base case: all dictionary entries as starting points
  SELECT 
    TARGET_SCHEMA AS origin_target_schema,
    TARGET_TABLE AS origin_target_table,
    SOURCE_SCHEMA,
    SOURCE_TABLE,
    1 AS depth
  FROM source_dict

  UNION ALL

  -- Recursive case: trace SOURCE_TABLE through dictionary when it's in ORADMART1
  SELECT 
    l.origin_target_schema,
    l.origin_target_table,
    d.SOURCE_SCHEMA,
    d.SOURCE_TABLE,
    l.depth + 1
  FROM lineage l
  JOIN source_dict d
    ON l.SOURCE_TABLE = d.TARGET_TABLE 
    AND l.SOURCE_SCHEMA = d.TARGET_SCHEMA
  WHERE l.SOURCE_SCHEMA = 'ORADMART1'
    AND l.depth < 5  -- prevent infinite loops
)
SELECT DISTINCT
  origin_target_schema AS TARGET_SCHEMA,
  origin_target_table AS TARGET_TABLE,
  SOURCE_SCHEMA,
  SOURCE_TABLE
FROM lineage
WHERE SOURCE_SCHEMA != 'ORADMART1'  -- only keep real sources (not intermediate DWH tables)
ORDER BY TARGET_SCHEMA, TARGET_TABLE, SOURCE_SCHEMA, SOURCE_TABLE

-- select distinct source_schema from dataplatform01_modelling_dev_catalog_01.bo_universes_excel_sheet_imports.mi_dwh_data_dictionary

-- ORADMART1 =>  MI DWH
-- ORFM => finance
-- ORFP => finance
-- CREATED AND DROPPED => (DROP)
-- null => (DROP)
-- SYS => (DROP)

In [0]:
%sql
    
CREATE OR REPLACE TABLE dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.MI_Dictionary_SAP_BO AS 
WITH RECURSIVE source_dict AS (
  SELECT DISTINCT
    upper(trim(TARGET_SCHEMA)) as TARGET_SCHEMA,
    upper(trim(TARGET_TABLE)) as TARGET_TABLE,
    upper(trim(SOURCE_SCHEMA)) as SOURCE_SCHEMA,
    upper(trim(SOURCE_TABLE)) as SOURCE_TABLE
  FROM dataplatform01_modelling_dev_catalog_01.bo_universes_excel_sheet_imports.mi_dwh_data_dictionary
  WHERE SOURCE_SCHEMA IS NOT NULL 
    AND upper(trim(SOURCE_SCHEMA)) NOT IN ('CREATED AND DROPPED', 'SYS')
    AND TARGET_TABLE IS NOT NULL
    AND upper(trim(TARGET_SCHEMA)) NOT IN ('CREATED AND DROPPED')
),
lineage AS (
  -- Base case: all dictionary entries as starting points
  SELECT 
    TARGET_SCHEMA AS origin_target_schema,
    TARGET_TABLE AS origin_target_table,
    SOURCE_SCHEMA,
    SOURCE_TABLE,
    1 AS depth
  FROM source_dict

  UNION ALL

  -- Recursive case: trace SOURCE_TABLE through dictionary when it's in ORADMART1
  SELECT 
    l.origin_target_schema,
    l.origin_target_table,
    d.SOURCE_SCHEMA,
    d.SOURCE_TABLE,
    l.depth + 1
  FROM lineage l
  JOIN source_dict d
    ON l.SOURCE_TABLE = d.TARGET_TABLE 
    AND l.SOURCE_SCHEMA = d.TARGET_SCHEMA
  WHERE l.SOURCE_SCHEMA = 'ORADMART1'
    AND l.depth < 5  -- prevent infinite loops
)
SELECT DISTINCT
  origin_target_schema AS TARGET_SCHEMA,
  origin_target_table AS TARGET_TABLE,
  SOURCE_SCHEMA,
  SOURCE_TABLE
FROM lineage
WHERE SOURCE_SCHEMA != 'ORADMART1'  -- only keep real sources (not intermediate DWH tables)
ORDER BY TARGET_SCHEMA, TARGET_TABLE, SOURCE_SCHEMA, SOURCE_TABLE

In [0]:
%sql
create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.Table_linage as
select 
  upper(trim(v3.BO_TABLE)) as BO_SQL_TABLE, 
  upper(trim(v3.SOURCE_TABLE)) as MI_Table, 
  upper(trim(v3.SCHEMA)) as MI_SCHEMA, 
  upper(trim(v3.SOURCE)) as MI_SOURCE, 
  upper(trim(d1.SOURCE_TABLE)) as Src_table, 
  upper(trim(d1.SOURCE_SCHEMA)) as Src_schema 
from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.BOSQL_MI_Mapping_V3 as V3
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.MI_Dictionary_SAP_BO as D1
on upper(trim(v3.SOURCE_TABLE)) = upper(trim(D1.TARGET_TABLE)) 
   and upper(trim(v3.SCHEMA)) = upper(trim(D1.TARGET_SCHEMA))
order by upper(trim(v3.BO_TABLE)) asc, upper(trim(v3.SOURCE_TABLE)) asc
-- where D1.TARGET_TABLE is null

In [0]:
%sql 

create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.applications_databrickUC_Schema_mapping  AS
SELECT 'ORACLE FINANCE TABLE' AS BO_SOURCE, 'AP' AS BO_source_schema, 'bronze_raw_orf_ap' AS Databricks_Schema
UNION ALL
SELECT 'ORACLE FINANCE TABLE', 'APPLSYS', 'bronze_raw_orf_applsys'
UNION ALL
SELECT 'ORACLE FINANCE TABLE', 'APPS', 'bronze_raw_orf_apps'
UNION ALL
SELECT 'ORACLE FINANCE TABLE', 'AR', 'bronze_raw_orf_ar'
UNION ALL
SELECT 'ORACLE FINANCE TABLE', 'CUST', 'bronze_raw_orf_cust'
UNION ALL
SELECT 'ORACLE FINANCE TABLE', 'FA', 'bronze_raw_orf_fa'
UNION ALL
SELECT 'ORACLE FINANCE TABLE', 'GL', 'bronze_raw_orf_gl'
UNION ALL
SELECT 'ORACLE FINANCE TABLE', 'HR', 'bronze_raw_orf_hr'
UNION ALL
SELECT 'ORACLE FINANCE TABLE', 'INV', 'bronze_raw_orf_inv'
UNION ALL
SELECT 'ORACLE FINANCE TABLE', 'JTF', 'bronze_raw_orf_jtf'
UNION ALL
SELECT 'ORACLE FINANCE TABLE', 'ORABOFP', 'bronze_raw_orf_orabofp'
UNION ALL
SELECT 'SYMPHONY', 'ORABUP0', 'bronze_raw_sym_orabup0'
UNION ALL
SELECT 'DW', 'ORABUP0', 'bronze_raw_symq_orabup0'
UNION ALL
SELECT 'DW', 'ORADMART1', 'bronze_raw_symq_oradmart1'
UNION ALL
SELECT 'DW', 'ORFM', NULL
UNION ALL
SELECT 'DW', 'ORFP', NULL
UNION ALL
SELECT 'ORACLE FINANCE TABLE', 'PO', 'bronze_raw_orf_po'
;

In [0]:
%sql
    
    
create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage as
SELECT distinct
  upper(trim(aws.Document_Id)) as Document_Id,
  upper(trim(aws.Document_name)) as Document_name,
  wcd.cluster,
  upper(trim(aws.source_DB_connection)) as SAP_BO_Connection,
  upper(trim(aws.table_Name)) as BO_TABLE,
  upper(trim(aws.schema_Name)) as BO_SCHEMA,
  upper(trim(tl.MI_Table)) as MI_Table,
  upper(trim(tl.MI_SCHEMA)) as MI_SCHEMA,
  upper(trim(tl.MI_SOURCE)) as MI_SOURCE,
  upper(trim(tl.Src_table)) as Src_table,
  upper(trim(tl.Src_schema)) as Src_schema,
  COALESCE(upper(trim(tl.Src_table)), upper(trim(tl.MI_Table)), upper(trim(aws.table_Name))) as Calc_source_table,
  (CASE 
    WHEN upper(trim(tl.Src_table)) IS NULL AND upper(trim(tl.MI_Table)) IS NULL THEN upper(trim(aws.schema_Name))
    WHEN upper(trim(tl.Src_table)) IS NULL THEN upper(trim(tl.MI_SCHEMA))
    ELSE upper(trim(tl.Src_schema))
  END) as Calc_source_schema,
  upper(trim(aws.Document_CUID)) as Document_CUID,
  upper(trim(aws.Full_path)) as Full_path,
  upper(trim(aws.sql_table)) as sql_table,
  upper(trim(aws.updated_by)) as updated_by,
--   (case when upper(trim(DB_In.tableName)) is null then 'N' else 'Y' end) as Databrick_ingested,
  current_timestamp() AS linage_ingestion_ts
FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source AS aws
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_cluster_details as wcd
  ON upper(trim(aws.Document_Id)) = upper(trim(wcd.Document_Id))
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms as cms
  ON upper(trim(aws.Document_Id)) = upper(trim(cms.Document_Id))
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.Table_linage AS tl
  ON upper(trim(aws.table_Name)) = upper(trim(tl.BO_SQL_TABLE))

In [0]:
%sql
    
    
ALTER TABLE dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage ADD COLUMN BO_DataConnection STRING;

MERGE INTO dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage AS awfl
USING (
  SELECT
    upper(trim(Document_Id)) AS Document_Id,
    concat_ws('|', array_sort(collect_set(upper(trim(Connection_Name))))) AS BO_DataConnection
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
  GROUP BY upper(trim(Document_Id))
) AS conn
ON upper(trim(awfl.Document_Id)) = conn.Document_Id
WHEN MATCHED THEN UPDATE SET awfl.BO_DataConnection = conn.BO_DataConnection;

In [0]:
%sql
create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage_UCflagged as 
SELECT distinct
 fl1.*, DB_schema.Databricks_Schema, (case when s1.table_schema is null or s1.table_name is null  then 'N' else 'Y' end) as databricks_ingested
from dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage as fl1
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.applications_databrickUC_Schema_mapping as DB_schema
  ON 
    (
      upper(trim(fl1.Calc_source_schema)) = upper(trim(DB_schema.BO_source_schema))
      AND (
        upper(trim(fl1.Calc_source_schema)) <> 'ORABUP0'
        OR (
          upper(trim(fl1.Calc_source_schema)) = 'ORABUP0'
          AND upper(trim(split(fl1.MI_SOURCE, ' ')[0])) = upper(trim(DB_schema.BO_SOURCE))
        )
      )
    )
left join (SELECT distinct table_schema, table_name FROM dataplatform01_central_prd_catalog_01.information_schema.tables ) as s1
on upper(trim(DB_schema.Databricks_Schema))=upper(Trim(s1.table_schema)) and upper(trim(fl1.Calc_source_table))=upper(trim(s1.table_name))




In [0]:
%sql
select cluster, count(distinct Document_Id) as Reports_cnt from dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage
group by cluster
order by cluster asc


select Databricks_Schema, databricks_ingested, count(distinct Calc_source_schema, Calc_source_table) as tables 
from dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage_UCflagged
where Databricks_Schema is not null 
and Calc_source_schema not in ('NOT AVAILABLE')
group by Databricks_Schema, databricks_ingested
order by Databricks_Schema asc, databricks_ingested asc

In [0]:
%sql 
SELECT DISTINCT BO_DataConnection, MI_SOURCE FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage
-- where Calc_source_schema = 'ORADMART1'
-- where Document_Id = '10040371'

In [0]:
%sql
    
select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage 
where Document_Id='221408'
where BO_TABLE = 'TBOR_NON_NCM_ORGANISATIONS'
order by Document_Id asc 
-- MI_Table asc, MI_SCHEMA asc, Src_schema asc, Src_table asc 

select * from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.MI_Dictionary_SAP_BO
where target_table = 'TBOR_NON_NCM_ORGANISATIONS'

select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.Table_linage
select 'V3' as category, distinct universe from dataplatform01_modelling_dev_catalog_01.bo_universes_excel_sheet_imports.v3_extraction
union 
select 'V3_remaining' as category, distinct universe from dataplatform01_modelling_dev_catalog_01.bo_universes_excel_sheet_imports.v3_extraction_remaining
where  table in ('DD_DEBTOR','DD_CLAIMSUNIT','FT_PAYMENT','DD_CURRENCY','DD_CUSTOMER','DD_CASE','T_DAILYCONV','FT_FINELEMDEBT','DD_FINELEMENTTYPE')

select distinct 'V3' as category, universe from dataplatform01_modelling_dev_catalog_01.bo_universes_excel_sheet_imports.v3_extraction
union all
select distinct 'V3_remaining' as category, universe from dataplatform01_modelling_dev_catalog_01.bo_universes_excel_sheet_imports.v3_extraction_remaining
Union all
select distinct 'BO Reports' as category,  DataSource_Name from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
where Document_Id='221408'

SELECT
  'active_webi_full_linage' as source,
  count(*) as rows,
  count(distinct Src_table) as distinct_src
FROM
  dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage
WHERE
  BO_TABLE = 'TBOR_NON_NCM_ORGANISATIONS'
UNION ALL
SELECT
  'Table_linage' as source,
  count(*) as rows,
  count(distinct Src_table) as distinct_src
FROM
  dataplatform01_central_dev_catalog_01.custom_sap_bo.Table_linage
WHERE
  BO_SQL_TABLE = 'TBOR_NON_NCM_ORGANISATIONS'

select distinct Calc_source_schema, calc_source_table, src_schema, src_table, MI_Table, MI_SCHEMA, MI_SOURCE from dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage
where Calc_source_schema not in ('ORFP','ORFM','ORADMART1','SYS','PO','ORABUP0','ORABOFP','AR','AP',
'GL','APPLSYS','APPS','CUST','HR','INV','JTF','FA')





In [0]:
%sql
select distinct checks.*, cms.DataSource_Name, cms.DataSource_Type from (
    select distinct 
    upper(trim(M.Table_Name)) as Master_table,
    upper(Trim(M.`Impact_Y/N`)) as impact_flag,
    upper(trim(M.`Used_in_MI`)) as MI_flag,
    upper(trim(FL.MI_BO_Table)) as MI_BO_Table,
    upper(trim(FL.Document_Id)) as BO_report
    from  dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.arcaderisk_master_tables as M
    left join (
    select case when Calc_source_table is null then BO_TABLE else Calc_source_table end as MI_BO_Table, Document_Id  from dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage) as FL
    on upper(trim(M.Table_Name)) = upper(trim(FL.MI_BO_Table))
    where upper(trim(M.`Impact_Y/N`)) = 'Y' 
    and upper(trim(M.`Used_in_MI`)) not in ('Y')
    and FL.MI_BO_Table is not null
) as checks
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms as cms
on upper(trim(checks.BO_report)) = upper(trim(cms.Document_Id))
;
select 
-- Master_table, 
distinct BO_report from _sqldf
-- group by Master_table

In [0]:
%sql
select distinct fl1.BO_SCHEMA, BO_TABLE,  Calc_source_table, cms.Connection_Name, cms.DataSource_Type, cms.DataSource_Name  from dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage as fl1
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms as cms
on upper(trim(fl1.Document_Id)) = upper(trim(cms.Document_Id))
where fl1.Calc_source_table is null and cms.DataSource_Type in ('fhsql')
and fl1.BO_TABLE is not null and fl1.BO_TABLE <> '' 
order by fl1.BO_SCHEMA asc, BO_TABLE asc



In [0]:
%sql
select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.Table_linage
where MI_Table in('VWMI_LATEST_FINDATA')

In [0]:
%sql
SHOW TABLES IN dataplatform01_central_prd_catalog_01.bronze_raw_sap_bo;

-- select * from  dataplatform01_central_prd_catalog_01.bronze_raw_sl1_symphony.tbor_countries limit 10
select distinct MI_SOURCE, Calc_source_schema from dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage
order by Calc_source_schema asc
where Databrick_ingested = 'N' and Calc_source_schema ='ORABUP0'

SELECT distinct table_schema, table_name FROM dataplatform01_central_prd_catalog_01.information_schema.tables 
-- WHERE table_schema = 'bronze_scd1_sym_orabup0'

select distinct t.SCHEMA, t.SOURCE from 
(  SELECT DISTINCT
    upper(trim(table)) as BO_TABLE,
    CASE 
      WHEN upper(SOURCE) LIKE 'ALIAS%' AND base_table IS NOT NULL AND base_table != '' 
      THEN upper(trim(base_table)) 
      ELSE upper(trim(TABLE)) 
    END AS SOURCE_TABLE,
    upper(trim(SCHEMA)) as SCHEMA,
    CASE
      WHEN upper(SOURCE) LIKE 'ALIAS%'
      THEN upper(trim(regexp_replace(regexp_replace(SOURCE, '(?i)^Alias\\s*', ''), '[()\\[\\]{}]', '')))
      ELSE upper(trim(SOURCE))
    END AS SOURCE
  FROM dataplatform01_modelling_dev_catalog_01.bo_universes_excel_sheet_imports.v3_extraction
  UNION
  SELECT DISTINCT
    upper(trim(table)) as BO_TABLE,
    CASE 
      WHEN upper(SOURCE) LIKE 'ALIAS%' AND base_table IS NOT NULL AND base_table != '' 
      THEN upper(trim(base_table)) 
      ELSE upper(trim(TABLE)) 
    END AS SOURCE_TABLE,
    upper(trim(SCHEMA)) as SCHEMA,
    CASE
      WHEN upper(SOURCE) LIKE 'ALIAS%'
      THEN upper(trim(regexp_replace(regexp_replace(SOURCE, '(?i)^Alias\\s*', ''), '[()\\[\\]{}]', '')))
      ELSE upper(trim(SOURCE))
    END AS SOURCE
  FROM dataplatform01_modelling_dev_catalog_01.bo_universes_excel_sheet_imports.v3_extraction_remaining
) as t